# Feature Preprocessing

This notebook prepares the consolidated customer features and churn labels for machine learning model development.

The preprocessing pipeline separates predictive features from the target, handles missing values appropriately, and prepares a leakage-safe dataset for model comparison.

In [12]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split

In [2]:
DATA_DIR = Path("../data/interim")

features = pd.read_csv(
    DATA_DIR / "customer_features.csv"
)

labels = pd.read_csv(
    "../data/interim/churn_labels.csv"
)

print("Features:", features.shape)
print("Labels:", labels.shape)

Features: (206209, 24)
Labels: (206185, 2)


# Feature Preprocessing

This notebook prepares the consolidated customer features and churn labels for machine learning.

The preprocessing workflow aligns the feature and target datasets, removes customers without valid labels, separates predictors from the target, validates feature types and ranges, and prepares a leakage-safe preprocessing pipeline for model comparison.

In [3]:
DATA_DIR = Path("../data/interim")

features = pd.read_csv(
    DATA_DIR / "customer_features.csv"
)

labels = pd.read_csv(
    DATA_DIR / "churn_labels.csv"
)

print("Features:", features.shape)
print("Labels:", labels.shape)

Features: (206209, 24)
Labels: (206185, 2)


In [4]:
model_data = features.merge(
    labels,
    on="user_id",
    how="inner",
    validate="one_to_one"
)

print("Model dataset:", model_data.shape)
print(
    "Unique customers:",
    model_data["user_id"].nunique()
)
print(
    "Duplicate user_id:",
    model_data["user_id"].duplicated().sum()
)

Model dataset: (206185, 25)
Unique customers: 206185
Duplicate user_id: 0


In [5]:
TARGET = "churn_like_label"
ID_COLUMN = "user_id"

X = model_data.drop(
    columns=[ID_COLUMN, TARGET]
)

y = model_data[TARGET].astype("int8")

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (206185, 23)
y shape: (206185,)


In [6]:
print(y.value_counts().sort_index())
print()
print(y.value_counts(normalize=True).sort_index())

churn_like_label
0    163024
1     43161
Name: count, dtype: int64

churn_like_label
0    0.790669
1    0.209331
Name: proportion, dtype: float64


In [7]:
X.dtypes

total_orders                   int64
max_order_number               int64
avg_days_between_orders      float64
latest_order_number            int64
latest_order_dow               int64
latest_order_hour              int64
median_purchase_interval     float64
min_purchase_interval        float64
max_purchase_interval        float64
std_purchase_interval        float64
prior_order_items              int64
prior_unique_products          int64
prior_reordered_items          int64
prior_orders                   int64
avg_items_per_order          float64
reorder_rate                 float64
products_per_order           float64
unique_aisles                  int64
unique_departments             int64
dominant_department_share    float64
dominant_aisle_share         float64
inactivity_gap               float64
inactivity_ratio             float64
dtype: object

In [8]:
print(
    "Non-numeric features:",
    X.select_dtypes(exclude=np.number).columns.tolist()
)

Non-numeric features: []


In [9]:
missing = X.isna().sum()

missing[missing > 0]

inactivity_ratio    82
dtype: int64

In [11]:
range_checks = {
    "reorder_rate": (
        X["reorder_rate"].min(),
        X["reorder_rate"].max()
    ),
    "dominant_department_share": (
        X["dominant_department_share"].min(),
        X["dominant_department_share"].max()
    ),
    "dominant_aisle_share": (
        X["dominant_aisle_share"].min(),
        X["dominant_aisle_share"].max()
    ),
}

range_checks

{'reorder_rate': (np.float64(0.0), np.float64(0.9895287958115184)),
 'dominant_department_share': (np.float64(0.095617529880478), np.float64(1.0)),
 'dominant_aisle_share': (np.float64(0.0415704387990762), np.float64(1.0))}

## Train / Validation / Test Split

The labelled customer dataset is divided into training, validation, and test subsets using stratified sampling.

A 70:15:15 split is used to preserve the churn-like class distribution across all subsets.

The split is performed before fitting any preprocessing transformation to prevent information leakage.

In [13]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

In [14]:
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

Train: (144329, 23) (144329,)
Validation: (30928, 23) (30928,)
Test: (30928, 23) (30928,)


In [15]:
split_distribution = pd.DataFrame({
    "Train": y_train.value_counts(normalize=True),
    "Validation": y_val.value_counts(normalize=True),
    "Test": y_test.value_counts(normalize=True)
}).sort_index()

split_distribution

,Train,Validation,Test
churn_like_label,,,
0,0.790666,0.790675,0.790675
1,0.209334,0.209325,0.209325


In [16]:
print(
    "Train ∩ Validation:",
    len(set(X_train.index) & set(X_val.index))
)

print(
    "Train ∩ Test:",
    len(set(X_train.index) & set(X_test.index))
)

print(
    "Validation ∩ Test:",
    len(set(X_val.index) & set(X_test.index))
)

Train ∩ Validation: 0
Train ∩ Test: 0
Validation ∩ Test: 0


In [17]:
print("Train target:")
print(y_train.value_counts().sort_index())

print("\nValidation target:")
print(y_val.value_counts().sort_index())

print("\nTest target:")
print(y_test.value_counts().sort_index())

Train target:
churn_like_label
0    114116
1     30213
Name: count, dtype: int64

Validation target:
churn_like_label
0    24454
1     6474
Name: count, dtype: int64

Test target:
churn_like_label
0    24454
1     6474
Name: count, dtype: int64


In [18]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

In [19]:
numeric_features = X_train.columns.tolist()

numeric_preprocessor = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

print("Numeric features:", len(numeric_features))
print(numeric_features)

Numeric features: 23
['total_orders', 'max_order_number', 'avg_days_between_orders', 'latest_order_number', 'latest_order_dow', 'latest_order_hour', 'median_purchase_interval', 'min_purchase_interval', 'max_purchase_interval', 'std_purchase_interval', 'prior_order_items', 'prior_unique_products', 'prior_reordered_items', 'prior_orders', 'avg_items_per_order', 'reorder_rate', 'products_per_order', 'unique_aisles', 'unique_departments', 'dominant_department_share', 'dominant_aisle_share', 'inactivity_gap', 'inactivity_ratio']


## Fit Preprocessing Pipeline

The preprocessing pipeline is fitted exclusively on the training data.

Median imputation handles missing numerical values, while standardization places features on a comparable scale for scale-sensitive models such as Logistic Regression.

Validation and test data are transformed using the preprocessing parameters learned from the training set only.

In [20]:
numeric_preprocessor.fit(X_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...), ('scaler', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](23,)","['total_orders','max_order_number','avg_days_between_orders',..., 'dominant_aisle_share','inactivity_gap','inactivity_ratio']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,23
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and "

In [21]:
X_train_processed = numeric_preprocessor.transform(X_train)
X_val_processed = numeric_preprocessor.transform(X_val)
X_test_processed = numeric_preprocessor.transform(X_test)

print("Processed Train:", X_train_processed.shape)
print("Processed Validation:", X_val_processed.shape)
print("Processed Test:", X_test_processed.shape)

Processed Train: (144329, 23)
Processed Validation: (30928, 23)
Processed Test: (30928, 23)


In [22]:
print(
    "Missing values after preprocessing:"
)

print(
    "Train:",
    np.isnan(X_train_processed).sum()
)

print(
    "Validation:",
    np.isnan(X_val_processed).sum()
)

print(
    "Test:",
    np.isnan(X_test_processed).sum()
)

Missing values after preprocessing:
Train: 0
Validation: 0
Test: 0


In [23]:
processed_summary = pd.DataFrame(
    X_train_processed,
    columns=numeric_features
).describe().loc[["mean", "std"]]

processed_summary

,total_orders,max_order_number,avg_days_between_orders,latest_order_number,latest_order_dow,latest_order_hour,median_purchase_interval,min_purchase_interval,max_purchase_interval,std_purchase_interval,...,prior_orders,avg_items_per_order,reorder_rate,products_per_order,unique_aisles,unique_departments,dominant_department_share,dominant_aisle_share,inactivity_gap,inactivity_ratio
mean,4.568615e-17,4.568615e-17,1.764431e-16,4.568615e-17,-7.483077e-17,5.051077e-17,7.207384e-17,3.741538e-18,3.087754e-16,-1.966277e-16,...,3.741538e-17,2.193723e-16,1.662031e-16,-2.150400e-16,-6.518154e-17,1.451323e-16,5.297231e-17,2.185846e-17,-7.955692e-17,2.944000e-17
std,1.000003e+00,1.000003e+00,1.000003e+00,1.000003e+00,1.000003e+00,1.000003e+00,1.000003e+00,1.000003e+00,1.000003e+00,1.000003e+00,...,1.000003e+00,1.000003e+00,1.000003e+00,1.000003e+00,1.000003e+00,1.000003e+00,1.000003e+00,1.000003e+00,1.000003e+00,1.000003e+00


In [24]:
print(
    "Training imputer statistics:",
    numeric_preprocessor.named_steps["imputer"].statistics_.shape
)

print(
    "Training scaler means:",
    numeric_preprocessor.named_steps["scaler"].mean_.shape
)

Training imputer statistics: (23,)
Training scaler means: (23,)


In [26]:
scaled_train = pd.DataFrame(
    X_train_processed,
    columns=numeric_features
)

scaled_train.describe().loc[["mean", "std"]]

,total_orders,max_order_number,avg_days_between_orders,latest_order_number,latest_order_dow,latest_order_hour,median_purchase_interval,min_purchase_interval,max_purchase_interval,std_purchase_interval,...,prior_orders,avg_items_per_order,reorder_rate,products_per_order,unique_aisles,unique_departments,dominant_department_share,dominant_aisle_share,inactivity_gap,inactivity_ratio
mean,4.568615e-17,4.568615e-17,1.764431e-16,4.568615e-17,-7.483077e-17,5.051077e-17,7.207384e-17,3.741538e-18,3.087754e-16,-1.966277e-16,...,3.741538e-17,2.193723e-16,1.662031e-16,-2.150400e-16,-6.518154e-17,1.451323e-16,5.297231e-17,2.185846e-17,-7.955692e-17,2.944000e-17
std,1.000003e+00,1.000003e+00,1.000003e+00,1.000003e+00,1.000003e+00,1.000003e+00,1.000003e+00,1.000003e+00,1.000003e+00,1.000003e+00,...,1.000003e+00,1.000003e+00,1.000003e+00,1.000003e+00,1.000003e+00,1.000003e+00,1.000003e+00,1.000003e+00,1.000003e+00,1.000003e+00


In [27]:
import joblib

In [28]:
MODEL_DIR = Path("../models")
ARTIFACT_DIR = Path("../artifacts")

MODEL_DIR.mkdir(exist_ok=True)
ARTIFACT_DIR.mkdir(exist_ok=True)

In [29]:
joblib.dump(
    numeric_preprocessor,
    ARTIFACT_DIR / "numeric_preprocessor.joblib"
)

['..\\artifacts\\numeric_preprocessor.joblib']

In [30]:
pd.DataFrame(
    X_train_processed,
    columns=numeric_features
).to_csv(
    ARTIFACT_DIR / "X_train_processed.csv",
    index=False
)

pd.DataFrame(
    X_val_processed,
    columns=numeric_features
).to_csv(
    ARTIFACT_DIR / "X_val_processed.csv",
    index=False
)

pd.DataFrame(
    X_test_processed,
    columns=numeric_features
).to_csv(
    ARTIFACT_DIR / "X_test_processed.csv",
    index=False
)

y_train.to_csv(
    ARTIFACT_DIR / "y_train.csv",
    index=False
)

y_val.to_csv(
    ARTIFACT_DIR / "y_val.csv",
    index=False
)

y_test.to_csv(
    ARTIFACT_DIR / "y_test.csv",
    index=False
)

print("Model-ready artifacts exported successfully.")

Model-ready artifacts exported successfully.


In [31]:
print("Train:", X_train_processed.shape, y_train.shape)
print("Validation:", X_val_processed.shape, y_val.shape)
print("Test:", X_test_processed.shape, y_test.shape)

print("\nArtifacts:")
for path in sorted(ARTIFACT_DIR.iterdir()):
    print("-", path.name)

Train: (144329, 23) (144329,)
Validation: (30928, 23) (30928,)
Test: (30928, 23) (30928,)

Artifacts:
- numeric_preprocessor.joblib
- X_test_processed.csv
- X_train_processed.csv
- X_val_processed.csv
- y_test.csv
- y_train.csv
- y_val.csv
